# Lab 6 — End-to-end Azure AI Search RAG evaluation

**RAG** = search your own documents first, then let the model answer using only what was found.

This lab evaluates the live pipeline, not prepared context: Azure AI Search semantic retrieval → a Foundry model answer → deterministic retrieval, answer-part, and citation checks → Foundry groundedness and response-completeness evaluation.

Because it runs live, results vary between runs. The question to keep in mind: when an answer is wrong, was it **Search** returning the wrong pages, or the **model** misusing good ones? That is why retrieval and generation are measured — and timed — separately.

The versioned dataset and metric helpers are shared with Lab 7 so the two retrieval approaches use the same contract.


## What is measured

The first three answer *did Search find the right pages?*; the next five answer *did the model use them properly?*; the last two are what it cost. Only groundedness and completeness need an AI judge — the rest are exact checks in plain Python.

- **Retrieval recall:** required procedure pages found in the untouched top five and in the deeper bounded candidate set.
- **Variant precision and rank:** expected-procedure-page precision@5, top-1 match, and reciprocal rank over the untouched top five.
- **Generation-context recall:** required procedure pages retained after procedure-variant isolation.
- **Citation coverage:** factual answer units containing at least one citation.
- **Citation validity:** citations that resolve to a supplied Search result.
- **Answer-part coverage:** every versioned required part is answered or explicitly marked unsupported.
- **Groundedness:** Foundry `builtin.groundedness` over the actual retrieved context and generated answer — catches invented facts.
- **Response completeness:** Foundry `builtin.response_completeness` against the independent ground truth — catches missing facts.
- **Latency:** Search, generation, and total wall-clock milliseconds.
- **Cost:** observable model tokens and semantic requests; optional USD estimates use explicit subscription-specific rates from `.env`.

Current APIs: [Azure AI Search semantic ranking](https://learn.microsoft.com/azure/search/semantic-how-to-query-request), [Foundry Responses API](https://learn.microsoft.com/azure/foundry/agents/quickstarts/responses-api), and [Foundry cloud evaluation](https://learn.microsoft.com/azure/foundry/how-to/develop/cloud-evaluation).


In [ ]:
import json
import os
import re
import sys
import time
from importlib.metadata import version
from pathlib import Path
from statistics import median

from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper = candidate / 'labs' / 'observability-and-evaluation' / 'rag_eval_utils.py'
    if helper.exists():
        repo_root = candidate
        sys.path.insert(0, str(helper.parent))
        break
else:
    raise FileNotFoundError('rag_eval_utils.py not found')

from rag_eval_utils import (
    document_evidence_keys,
    estimate_cost_usd,
    load_cases,
    prepare_coherent_search_context,
    price_rates_from_env,
    primitive,
    ranked_evidence_metrics,
    render_cited_answer,
    response_token_usage,
    retrieval_recall,
    validate_cited_answer,
)

load_dotenv(repo_root / '.env')
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
evaluator_model = os.getenv('RAG_EVALUATOR_MODEL') or model_deployment
judge_repeat_count = int(os.getenv('RAG_EVAL_JUDGE_REPEATS', '3'))
if not 3 <= judge_repeat_count <= 9 or judge_repeat_count % 2 == 0:
    raise ValueError('RAG_EVAL_JUDGE_REPEATS must be an odd number between 3 and 9')
search_candidate_cap = int(os.getenv('RAG_SEARCH_CANDIDATE_CAP', '10'))
if not 5 <= search_candidate_cap <= 50:
    raise ValueError('RAG_SEARCH_CANDIDATE_CAP must be between 5 and 50')
search_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
search_index = os.getenv('AZURE_SEARCH_INDEX') or os.getenv('LAB_SEARCH_INDEX')
raw_namespace = (
    os.getenv('WORKSHOP_RESOURCE_NAMESPACE')
    or os.getenv('WORKSHOP_TEAM_ID')
    or os.getenv('WORKSHOP_PARTICIPANT_ID')
    or ''
)
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not all((endpoint, model_deployment, search_endpoint, search_index, resource_namespace)):
    raise ValueError('Missing Foundry, Search, model, or WORKSHOP_RESOURCE_NAMESPACE configuration')
print({'namespace': resource_namespace, 'index': search_index, 'generator_model': model_deployment, 'evaluator_model': evaluator_model, 'judge_repeats': judge_repeat_count, 'search_candidate_cap': search_candidate_cap, 'azure-ai-projects': version('azure-ai-projects')})

In [ ]:
dataset_path = repo_root / 'labs' / 'observability-and-evaluation' / 'data' / 'rag-evaluation-cases-v2.json'
dataset, search_cases = load_cases(dataset_path, 'azure_ai_search')
assert dataset['dataset_id'] == 'synthetic-grid-rag-e2e-v2'
assert dataset['answer_contract_version'] == 'search-required-parts-v2'
assert len(search_cases) >= 3
print({'dataset': dataset['dataset_id'], 'answer_contract': dataset['answer_contract_version'], 'cases': [row['case_id'] for row in search_cases]})

## Execute the live Search → answer path

Per case: search → score the raw ranking → keep only the top result's procedure variant → generate a cited answer → validate and time it. The variant filter matters because the index holds near-identical procedures, and blending controls from two of them would be dangerous.

Every retrieved page receives a stable `[S#]` label. Required-page recall is reported for the untouched top five and for a bounded candidate set (10 by default). Expected-page precision@5, top-1 match, and reciprocal rank stay tied to the untouched top five. Answer generation keeps only candidate documents sharing the top-ranked result's `vwi_code`; source IDs are rebuilt for that coherent subset. A separate generation-context recall gate proves that filtering retained every expected procedure page. The groundedness evaluator receives that exact coherent source set.




In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import InteractiveBrowserCredential
from azure.search.documents import SearchClient

credential = InteractiveBrowserCredential()
search_client = SearchClient(endpoint=search_endpoint, index_name=search_index, credential=credential)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()
rates = price_rates_from_env()


def generate_cited_search_answer(case: dict, context: str, valid_source_ids: set[str]):
    query = case['query']
    required_answer_parts = case['required_answer_parts']
    required_part_ids = [part['id'] for part in required_answer_parts]
    required_parts_text = chr(10).join(
        f"- {part['id']}: {part['description']}" for part in required_answer_parts
    )
    ordered_ids = sorted(valid_source_ids, key=lambda value: int(value[1:]))
    schema = {
        'type': 'object',
        'properties': {
            'claims': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'answer_part_id': {'type': 'string', 'enum': required_part_ids},
                        'text': {'type': 'string'},
                        'source_ids': {
                            'type': 'array',
                            'items': {'type': 'string', 'enum': ordered_ids},
                        },
                    },
                    'required': ['answer_part_id', 'text', 'source_ids'],
                    'additionalProperties': False,
                },
            },
            'insufficient_evidence': {
                'type': 'array',
                'items': {'type': 'string', 'enum': required_part_ids},
            },
        },
        'required': ['claims', 'insufficient_evidence'],
        'additionalProperties': False,
    }
    response = openai_client.responses.create(
        model=model_deployment,
        instructions=(
            'Answer only from the supplied Search sources and cover exactly the listed required '
            'answer parts. Return one independent, single-sentence factual claim per claims item. '
            'Tag every claim with exactly one required answer-part ID and cite every supporting '
            'source ID. Every required answer part must have at least one claim or be listed once '
            'in insufficient_evidence, never both. Use insufficient_evidence only when no supplied '
            'source can support that required part. Do not introduce scope, exclusivity, or '
            'completeness requirements outside the listed parts, combine controls from similarly '
            'named procedure variants, compress independently supported controls into an uncited '
            'summary, invent facts, or use IDs outside their enums.'
        ),
        input=(
            f"Question: {query}\n\n"
            f"Required answer parts (contract search-required-parts-v2):\n{required_parts_text}\n\n"
            f"Search sources:\n{context}"
        ),
        text={
            'format': {
                'type': 'json_schema',
                'name': 'cited_search_answer',
                'strict': True,
                'schema': schema,
            }
        },
        max_output_tokens=1200,
    )
    validation = validate_cited_answer(
        json.loads(response.output_text),
        valid_source_ids,
        required_answer_part_ids=required_part_ids,
    )
    if validation['valid_coverage'] != 1.0 or validation['validity'] != 1.0:
        raise ValueError(f'Generated answer failed deterministic citation validation: {validation}')
    return response, validation, render_cited_answer(validation)


search_rows = []
current_search_case_diagnostic = None
for case in search_cases:
    required_part_ids = [part['id'] for part in case['required_answer_parts']]
    current_search_case_diagnostic = {
        'case_id': case['case_id'],
        'answer_contract_version': dataset['answer_contract_version'],
        'required_answer_part_ids': required_part_ids,
        'answered_answer_part_ids': [],
        'answer_part_coverage': None,
        'insufficient_evidence': [],
        'stage': 'retrieval',
    }
    total_started = time.perf_counter()
    retrieval_started = time.perf_counter()
    documents = [
        dict(result)
        for result in search_client.search(
            search_text=case['query'],
            query_type='semantic',
            semantic_configuration_name='default',
            top=search_candidate_cap,
            select=['id', 'vwi_code', 'title', 'excerpt', 'content', 'source_file', 'page_number'],
        )
    ]
    retrieval_ms = (time.perf_counter() - retrieval_started) * 1000
    if not documents:
        raise RuntimeError(f"{case['case_id']}: Search returned no documents")

    top_five_documents = documents[:5]
    top_five_keys = set().union(
        *(document_evidence_keys(doc) for doc in top_five_documents)
    )
    top_five_recall = retrieval_recall(
        case['expected_evidence_groups'], top_five_keys
    )
    candidate_keys = set().union(
        *(document_evidence_keys(doc) for doc in documents)
    )
    candidate_recall = retrieval_recall(
        case['expected_evidence_groups'], candidate_keys
    )
    raw_ranked_metrics = ranked_evidence_metrics(
        case['expected_evidence_groups'], top_five_documents
    )
    prepared_context = prepare_coherent_search_context(documents)
    generation_documents = prepared_context['documents']
    primary_vwi_code = prepared_context['primary_vwi_code']
    context = prepared_context['context']
    valid_source_ids = prepared_context['valid_source_ids']
    generation_keys = set().union(
        *(document_evidence_keys(doc) for doc in generation_documents)
    )
    generation_recall = retrieval_recall(
        case['expected_evidence_groups'], generation_keys
    )
    raw_ranked_sources = [
        {
            'rank': rank,
            'id': str(document.get('id') or ''),
            'vwi_code': str(document.get('vwi_code') or ''),
        }
        for rank, document in enumerate(documents, start=1)
    ]
    generation_sources = [
        {
            'source_id': f'S{rank}',
            'id': str(document.get('id') or ''),
            'vwi_code': str(document.get('vwi_code') or ''),
        }
        for rank, document in enumerate(generation_documents, start=1)
    ]
    current_search_case_diagnostic.update({
        'retrieval_recall_at_5': top_five_recall['recall'],
        'retrieval_recall_at_candidate_cap': candidate_recall['recall'],
        'retrieval_candidate_cap': search_candidate_cap,
        'variant_precision_at_5': raw_ranked_metrics['precision'],
        'top_1_variant_match': raw_ranked_metrics['top_1_match'],
        'variant_first_match_rank': raw_ranked_metrics['first_match_rank'],
        'variant_reciprocal_rank': raw_ranked_metrics['reciprocal_rank'],
        'generation_context_recall': generation_recall['recall'],
        'primary_vwi_code': primary_vwi_code,
        'raw_ranked_sources': raw_ranked_sources,
        'generation_sources': generation_sources,
        'generation_documents': len(generation_documents),
    })
    generation_started = time.perf_counter()
    current_search_case_diagnostic['stage'] = 'generation'
    response, citations, answer = generate_cited_search_answer(
        case, context, valid_source_ids
    )
    generation_ms = (time.perf_counter() - generation_started) * 1000
    current_search_case_diagnostic.update({
        'response': answer,
        'answered_answer_part_ids': citations['answered_answer_part_ids'],
        'answer_part_coverage': citations['answer_part_coverage'],
        'insufficient_evidence': citations['insufficient_evidence'],
        'stage': 'validated',
    })

    usage = response_token_usage(response)
    cost = estimate_cost_usd(usage, rates)
    row = {
        'case_id': case['case_id'],
        'query': case['query'],
        'ground_truth': case['ground_truth'],
        'context': context,
        'response': answer,
        'retrieved_documents': len(documents),
        'generation_documents': len(generation_documents),
        'primary_vwi_code': primary_vwi_code,
        'raw_ranked_sources': raw_ranked_sources,
        'generation_sources': generation_sources,
        'retrieval_recall_at_5': top_five_recall['recall'],
        'recall_at_5_details': top_five_recall['groups'],
        'retrieval_recall_at_candidate_cap': candidate_recall['recall'],
        'retrieval_candidate_cap': search_candidate_cap,
        'candidate_recall_details': candidate_recall['groups'],
        'variant_precision_at_5': raw_ranked_metrics['precision'],
        'top_1_variant_match': raw_ranked_metrics['top_1_match'],
        'variant_first_match_rank': raw_ranked_metrics['first_match_rank'],
        'variant_reciprocal_rank': raw_ranked_metrics['reciprocal_rank'],
        'generation_context_recall': generation_recall['recall'],
        'generation_recall_details': generation_recall['groups'],
        'citation_coverage': citations['coverage'],
        'citation_validity': citations['validity'],
        'invalid_citations': citations['invalid_citations'],
        'insufficient_evidence': citations['insufficient_evidence'],
        'required_answer_part_ids': citations['required_answer_part_ids'],
        'answered_answer_part_ids': citations['answered_answer_part_ids'],
        'answer_part_coverage': citations['answer_part_coverage'],
        'answer_contract_version': dataset['answer_contract_version'],
        'retrieval_ms': round(retrieval_ms, 2),
        'generation_ms': round(generation_ms, 2),
        'total_ms': round((time.perf_counter() - total_started) * 1000, 2),
        **usage,
        **cost,
    }
    search_rows.append(row)
    current_search_case_diagnostic = None
    print(json.dumps({key: row[key] for key in (
        'case_id',
        'primary_vwi_code',
        'raw_ranked_sources',
        'generation_sources',
        'retrieval_recall_at_5',
        'retrieval_recall_at_candidate_cap',
        'retrieval_candidate_cap',
        'variant_precision_at_5',
        'top_1_variant_match',
        'variant_first_match_rank',
        'variant_reciprocal_rank',
        'generation_context_recall',
        'citation_coverage',
        'citation_validity',
        'answer_part_coverage',
        'required_answer_part_ids',
        'answered_answer_part_ids',
        'insufficient_evidence',
        'retrieval_ms',
        'generation_ms',
        'model_input_tokens',
        'model_output_tokens',
        'estimated_cost_usd',
    )}, indent=2))

assert len(search_rows) == len(search_cases)
assert all(row['context'] and row['response'] for row in search_rows)
print('PASS — every versioned case executed against live Search and the live model.')

## Evaluate groundedness and response completeness with judge consensus in Foundry

A judge is itself a language model and can score the same answer differently each time, so every case is sent several times and the median is used. `RAG_EVAL_JUDGE_REPEATS` must be odd so there is always a middle value.

This job submits the live `query`, exact coherent generation `context`, and frozen rendered `response` `RAG_EVAL_JUDGE_REPEATS` times (three by default). Groundedness uses that exact context; completeness uses the independent versioned ground truth. Every raw score and evaluator reason is retained; the mean is reported and the predeclared median is the consensus gate. The evaluator's own token use is reported separately because it is evaluation overhead, not application-path cost.


In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'eval_item_id': {'type': 'string'},
            'run_case_id': {'type': 'string'},
            'judge_repeat': {'type': 'integer'},
            'query': {'type': 'string'},
            'context': {'type': 'string'},
            'response': {'type': 'string'},
            'ground_truth': {'type': 'string'},
        },
        'required': ['eval_item_id', 'run_case_id', 'judge_repeat', 'query', 'context', 'response', 'ground_truth'],
    },
)
groundedness_criterion = TestingCriterionAzureAIEvaluator(
    type='azure_ai_evaluator',
    name='groundedness',
    evaluator_name='builtin.groundedness',
    initialization_parameters={'deployment_name': evaluator_model},
    data_mapping={
        'query': '{{item.query}}',
        'context': '{{item.context}}',
        'response': '{{item.response}}',
    },
)
completeness_criterion = TestingCriterionAzureAIEvaluator(
    type='azure_ai_evaluator',
    name='response_completeness',
    evaluator_name='builtin.response_completeness',
    initialization_parameters={'deployment_name': evaluator_model},
    data_mapping={
        'ground_truth': '{{item.ground_truth}}',
        'response': '{{item.response}}',
    },
)
run_suffix = str(int(time.time()))
eval_object = openai_client.evals.create(
    name=f'd2-e2e-search-{resource_namespace}-{run_suffix}',
    data_source_config=data_source_config,
    testing_criteria=[groundedness_criterion, completeness_criterion],
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f'd2-e2e-search-run-{resource_namespace}-{run_suffix}',
    metadata={'namespace': resource_namespace, 'dataset': dataset['dataset_id'], 'answer_contract': dataset['answer_contract_version'], 'target': 'azure-ai-search', 'judge_repeats': str(judge_repeat_count), 'evaluator_model': evaluator_model},
    data_source={
        'type': 'jsonl',
        'source': {
            'type': 'file_content',
            'content': [
                {
                    'item': {
                        'eval_item_id': f"{row['case_id']}-J{judge_repeat}",
                        'run_case_id': row['case_id'],
                        'judge_repeat': judge_repeat,
                        **{key: row[key] for key in ('query', 'context', 'response', 'ground_truth')},
                    }
                }
                for row in search_rows
                for judge_repeat in range(1, judge_repeat_count + 1)
            ],
        },
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('Search RAG evaluation exceeded 20 minutes')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)
output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
if eval_run.status != 'completed':
    run_error = getattr(eval_run, 'error', None)
    if hasattr(run_error, 'model_dump'):
        run_error = run_error.model_dump(mode='json')
    raise RuntimeError(f'Evaluation infrastructure failure: {{"status": {eval_run.status!r}, "eval_id": {eval_object.id!r}, "run_id": {eval_run.id!r}, "server_error": {run_error!r}, "output_items": {len(output_items)}, "report_url": {getattr(eval_run, "report_url", None)!r}}}')
expected_output_items = len(search_rows) * judge_repeat_count
output_deadline = time.monotonic() + 2 * 60
while len(output_items) < expected_output_items:
    if time.monotonic() > output_deadline:
        raise TimeoutError(f'Evaluation completed but exposed only {len(output_items)}/{expected_output_items} output items')
    time.sleep(2)
    output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
assert len(output_items) == expected_output_items
metrics = ('groundedness', 'response_completeness')
scores_by_case = {
    row['case_id']: {metric: {} for metric in metrics}
    for row in search_rows
}
reasons_by_case = {
    row['case_id']: {metric: {} for metric in metrics}
    for row in search_rows
}
for item in output_items:
    data = primitive(item)
    source_item = data['datasource_item']
    case_id = source_item['run_case_id']
    judge_repeat = int(source_item['judge_repeat'])
    for metric in metrics:
        result = next(result for result in data['results'] if result['name'] == metric)
        if result.get('error') or result.get('status') in ('failed', 'error', 'canceled'):
            raise RuntimeError(f'{case_id}-J{judge_repeat}: {metric} evaluator failed: {result}')
        if judge_repeat in scores_by_case[case_id][metric]:
            raise RuntimeError(f'{case_id}: duplicate {metric} judge repeat {judge_repeat}')
        reason = result.get('reason')
        if not isinstance(reason, str) or not reason.strip():
            raise RuntimeError(
                f'{case_id}-J{judge_repeat}: {metric} evaluator returned no reason'
            )
        scores_by_case[case_id][metric][judge_repeat] = float(result['score'])
        reasons_by_case[case_id][metric][judge_repeat] = reason.strip()
for row in search_rows:
    for metric in metrics:
        samples = [
            scores_by_case[row['case_id']][metric][repeat]
            for repeat in range(1, judge_repeat_count + 1)
        ]
        reasons = [
            reasons_by_case[row['case_id']][metric][repeat]
            for repeat in range(1, judge_repeat_count + 1)
        ]
        row[f'{metric}_samples'] = samples
        row[f'{metric}_reasons'] = reasons
        row[f'{metric}_mean'] = round(sum(samples) / len(samples), 3)
        row[metric] = float(median(samples))

evaluator_usage = [primitive(item) for item in (getattr(eval_run, 'per_model_usage', None) or [])]
print({
    'judge_consensus': {
        row['case_id']: {
            metric: {
                'samples': row[f'{metric}_samples'],
                'reasons': row[f'{metric}_reasons'],
                'mean': row[f'{metric}_mean'],
                'median': row[metric],
            }
            for metric in metrics
        }
        for row in search_rows
    },
    'answer_contract_version': dataset['answer_contract_version'],
    'evaluator_model': evaluator_model,
    'evaluator_usage': evaluator_usage,
    'report_url': getattr(eval_run, 'report_url', None),
})
assert len(scores_by_case) == len(search_rows)
print(f'PASS — Foundry returned {judge_repeat_count} groundedness and completeness judge samples with reasons and median consensus for every live RAG case.')

## Release view and success check

The thresholds below are an explicit starting policy, not a universal standard. A failed quality gate is useful evidence: inspect the row-level answer and retrieval details before changing prompts or indexes.

In [ ]:
thresholds = {
    'retrieval_recall_at_candidate_cap': 1.0,
    'generation_context_recall': 1.0,
    'variant_reciprocal_rank': 1.0,
    'citation_coverage': 1.0,
    'citation_validity': 1.0,
    'groundedness': 3.0,
    'response_completeness': 4.0,
}
for row in search_rows:
    deterministic_ok = all(
        row[metric] >= minimum
        for metric, minimum in thresholds.items()
        if metric not in ('groundedness', 'response_completeness')
    )
    ranking_ok = row['top_1_variant_match'] and row['variant_reciprocal_rank'] == 1.0
    judge_ok = all(
        row[metric] >= thresholds[metric]
        for metric in ('groundedness', 'response_completeness')
    )
    row['groundedness_pass_rate'] = round(
        sum(score >= thresholds['groundedness'] for score in row['groundedness_samples'])
        / len(row['groundedness_samples']),
        3,
    )
    row['response_completeness_pass_rate'] = round(
        sum(
            score >= thresholds['response_completeness']
            for score in row['response_completeness_samples']
        )
        / len(row['response_completeness_samples']),
        3,
    )
    parts_ok = row['answer_part_coverage'] == 1.0
    evidence_ok = not row['insufficient_evidence']
    row['deterministic_gate_passed'] = (
        deterministic_ok and ranking_ok and parts_ok and evidence_ok
    )
    row['release_gate_passed'] = row['deterministic_gate_passed'] and judge_ok
    print(json.dumps({
        'case_id': row['case_id'],
        'raw_ranked_sources': row['raw_ranked_sources'],
        'generation_sources': row['generation_sources'],
        'recall@5': row['retrieval_recall_at_5'],
        'candidate_recall': {
            'k': row['retrieval_candidate_cap'],
            'recall': row['retrieval_recall_at_candidate_cap'],
        },
        'variant_precision@5': row['variant_precision_at_5'],
        'top_1_variant_match': row['top_1_variant_match'],
        'variant_first_match_rank': row['variant_first_match_rank'],
        'variant_reciprocal_rank': row['variant_reciprocal_rank'],
        'generation_context_recall': row['generation_context_recall'],
        'citation_coverage': row['citation_coverage'],
        'citation_validity': row['citation_validity'],
        'groundedness_samples': row['groundedness_samples'],
        'groundedness_reasons': row['groundedness_reasons'],
        'groundedness_mean': row['groundedness_mean'],
        'groundedness_median': row['groundedness'],
        'groundedness_pass_rate': row['groundedness_pass_rate'],
        'response_completeness_samples': row['response_completeness_samples'],
        'response_completeness_reasons': row['response_completeness_reasons'],
        'response_completeness_mean': row['response_completeness_mean'],
        'response_completeness_median': row['response_completeness'],
        'response_completeness_pass_rate': row['response_completeness_pass_rate'],
        'answer_contract_version': row['answer_contract_version'],
        'required_answer_part_ids': row['required_answer_part_ids'],
        'answered_answer_part_ids': row['answered_answer_part_ids'],
        'answer_part_coverage': row['answer_part_coverage'],
        'insufficient_evidence': row['insufficient_evidence'],
        'latency_ms': {'retrieval': row['retrieval_ms'], 'generation': row['generation_ms'], 'total': row['total_ms']},
        'usage': {'input_tokens': row['model_input_tokens'], 'output_tokens': row['model_output_tokens'], 'semantic_requests': row['semantic_requests']},
        'estimated_cost_usd': row['estimated_cost_usd'],
        'release_gate_passed': row['release_gate_passed'],
    }, indent=2))

assert all(0.0 <= row['retrieval_recall_at_5'] <= 1.0 for row in search_rows)
assert all(0.0 <= row['retrieval_recall_at_candidate_cap'] <= 1.0 for row in search_rows)
assert all(row['retrieval_candidate_cap'] == search_candidate_cap for row in search_rows)
assert all(0.0 <= row['generation_context_recall'] <= 1.0 for row in search_rows)
assert all(0.0 <= row['variant_precision_at_5'] <= 1.0 for row in search_rows)
assert all(0.0 <= row['variant_reciprocal_rank'] <= 1.0 for row in search_rows)
assert all(0.0 <= row['citation_coverage'] <= 1.0 for row in search_rows)
assert all(row['answer_contract_version'] == 'search-required-parts-v2' for row in search_rows)
assert all(row['answer_part_coverage'] == 1.0 for row in search_rows)
assert all(row['retrieval_ms'] > 0 and row['total_ms'] >= row['retrieval_ms'] for row in search_rows)
assert all(row['groundedness'] >= 1.0 for row in search_rows)
assert all(row['response_completeness'] >= 1.0 for row in search_rows)
assert all(row['release_gate_passed'] for row in search_rows), 'One or more Search RAG release gates failed; inspect the deterministic and judge-consensus diagnostics above.'
print('SUCCESS CHECK — raw retrieval, coherent generation context, answer-part completeness, citations, judge reasons, groundedness, latency, and cost dimensions are populated.')

## Participant challenge

Add one representative procedure case with explicit `required_answer_parts` to `data/rag-evaluation-cases-v2.json`, rerun the lab, and explain whether a failure is caused by retrieval, generation, citation behavior, or the release threshold. Do not lower a threshold without a documented reason.

In [ ]:
# TODO: add a participant-owned case and compare its row-level metrics.
# Keep the case synthetic and give each required evidence group a stable identifier.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
if allow_cleanup:
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation and its run.')
else:
    print('Cleanup disabled so the Foundry evaluation report remains available.')